<a href="https://colab.research.google.com/github/victorlavrenko/answer-engineering/blob/main/notebooks/instruction-placement-reproduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instruction Duplication Demonstration

## Answer Engineering: Local Trajectory Control for Protocol-Constrained Reasoning

This notebook demonstrates the effect of repeating the system instruction
immediately after the user question.

For every selected clinical case, the same model, Answer Engineering rules,
system prompt, generation parameters, and dataset row are evaluated under two
conditions:

1. **System only**

```text
   System: [instruction]
   User:   [clinical question]
```

2. **Instruction duplicated after the query**

```text
   System: [instruction]
   User:   [clinical question]

           [same instruction]
```

The notebook evaluates both conditions on the same selected SSNHL and
conductive-control cases. Only the text of the user request changes.

The workflow is intentionally small:

1. configure the dataset and model
2. install and load Answer Engineering
3. extract the notebook-authored trajectory rules
4. select one fixed set of cases per clinical scope
5. generate answers under both instruction-placement conditions
6. compare accuracy and paired case-level changes

Case-level responses are also saved to a cumulative compressed JSONL file.
In Google Colab, the notebook requests a browser download after each completed
condition and once again at the end.

In [1]:
from __future__ import annotations

DATASET_ID = "lavrenko/casefactory"
SPLIT = "test"
MODEL_ID = "OpenMeditron/Meditron3-8B"
N_EVAL = 1000
MAX_NEW_TOKENS = 1024
VERBOSITY = 0
NOTEBOOK_NAME = "reproduce.ipynb"

## Answer Engineering Package Installation

In [2]:
import importlib.util
import os
from pathlib import Path

package = "answer_engineering"
repo = f"{package.replace('_', '-')}"

try:
    from google.colab import userdata  # type: ignore[reportMissingImports]

    t = os.getenv("GITHUB_TOKEN") or userdata.get("GITHUB_TOKEN")  # type: ignore[reportUnknownMemberType]
except Exception:
    t = os.getenv("GITHUB_TOKEN")

if Path(repo).is_dir():
    !git -C {repo} pull
elif Path.cwd().name != "notebooks":
    u = f"https://{t}@github.com/" if t else "https://github.com/"
    !git clone {u}victorlavrenko/{repo}.git
if importlib.util.find_spec(package) is None:
    target = f"{repo}[hf]" if Path(repo).is_dir() else "..[dev,hf]"
    %pip install -e {target}
    raise SystemExit("Restart runtime required")

Already up to date.


## Dataset setup

Instantiate and preload the dataset once here so rerunning the evaluation loop does not trigger a fresh dataset download.

In [3]:
from ae_paper_reproduction import CachedHFDataset, Dataset

dataset: Dataset = CachedHFDataset(DATASET_ID, SPLIT)
dataset.materialize()

README.md:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  368kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  276kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/4000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3000 [00:00<?, ? examples/s]

CachedHFDataset(dataset_id='lavrenko/casefactory', split='test', question_field='question', gold_field='gold', id_field='id', case_type_field='case_type', revision=None)

## Generation runtime setup

Instantiate and preload the model in its own cell so you can rerun it independently while editing the notebook.

In [4]:
from answer_engineering import GenerationRuntime

runtime = GenerationRuntime(MODEL_ID)
runtime.materialize()

config.json:   0%|          | 0.00/903 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

GenerationRuntime(model_id='OpenMeditron/Meditron3-8B', revision=None, device_map='auto', dtype='auto', trust_remote_code=False, show_hf_hub_progress_bars_on_load=True, show_hf_hub_progress_bars_on_generate=False)

## Preview clinical subruns

The rules cell below produces separate SSNHL and conductive subruns. Each
subrun will subsequently be evaluated under both instruction-placement
conditions.

In [5]:
from ae_paper_reproduction import NotebookSubruns

subruns = NotebookSubruns(NOTEBOOK_NAME, dataset=dataset, model=runtime)

print(f"Loaded {len(subruns)} subruns from {NOTEBOOK_NAME}:")
for subrun in subruns:
    print(f"- {subrun.name}")

Loaded 2 subruns from reproduce.ipynb:
- trajectory-editing-orl-ssnhl-acute
- trajectory-editing-orl-conductive-acute


## Evaluate both instruction-placement conditions

Each extracted clinical subrun selects its tasks once. The same tasks are then
evaluated twice: first with the instruction only in the system message, and
then with the identical instruction repeated after the clinical question.

Each completed case is appended immediately to `instruction-placement-results.jsonl`.
After each condition, the cumulative file is compressed and—when running in
Google Colab—sent to the browser with `google.colab.files.download`. Allow
multiple downloads from Colab if your browser asks.

In [6]:
import gzip
import json
import shutil
from pathlib import Path

from ae_paper_reproduction import (
    EvaluationPrinter,
    Progress,
    RulesetEvaluationResult,
    SubrunResult,
    SubrunTask,
)
from answer_engineering import (
    GenerationPolicy,
    GenerationRequest,
    GenerationResult,
)

request_variants = {
    "system-only": lambda task, subrun: task.question,
    "duplicated-after-query": (
        lambda task, subrun:
            task.question + "\n\n" + subrun.system_prompt
    ),
}

printer = EvaluationPrinter(verbosity=VERBOSITY)
results = {}

RESULTS_PATH = Path("instruction-placement-results.jsonl")

from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version
import platform
import subprocess

def package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None

try:
    repo_commit = subprocess.check_output(
        ["git", "-C", repo, "rev-parse", "HEAD"],
        text=True,
    ).strip()
except Exception:
    repo_commit = None

metadata = {
    "record_type": "metadata",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_id": DATASET_ID,
    "split": SPLIT,
    "model_id": MODEL_ID,
    "n_eval": N_EVAL,
    "max_new_tokens": MAX_NEW_TOKENS,
    "repo_commit": repo_commit,
    "python": platform.python_version(),
    "torch": package_version("torch"),
    "transformers": package_version("transformers"),
    "datasets": package_version("datasets"),
    "answer_engineering": package_version("answer-engineering"),
}
RESULTS_PATH.write_text(
    json.dumps(metadata, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

def save_result(subrun_name, variant, task_result):
    row = {
        "record_type": "result",
        "subrun": subrun_name,
        "condition": variant,
        "case_id": task_result.id,
        "case_type": task_result.case_type,
        "question": task_result.question,
        "gold": task_result.gold,
        "answer": task_result.answer,
        "ok": task_result.ok,
        "runtime_sec": task_result.runtime_sec,
    }
    with RESULTS_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()

def download_results(label):
    compressed = Path(f"instruction-placement-results-{label}.jsonl.gz")
    with RESULTS_PATH.open("rb") as source, gzip.open(compressed, "wb") as target:
        shutil.copyfileobj(source, target)

    try:
        from google.colab import files
        files.download(str(compressed))
    except ImportError:
        print(f"Saved {compressed.resolve()}")

for i, subrun in enumerate(subruns):
    tasks: tuple[SubrunTask, ...] = subrun.select_tasks(n=N_EVAL)

    policy = GenerationPolicy(
        rules=subrun.compiled_rules,
        system_prompt=subrun.system_prompt,
        verbosity=VERBOSITY,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    for variant, build_question in request_variants.items():
        run_name = f"{subrun.name}-{variant}"
        print(f"{i}. Evaluating {run_name}:")

        task_results: list[RulesetEvaluationResult] = []
        correct = 0
        progress = Progress(tasks, desc=run_name)

        try:
            for task in progress:
                printer.task_start(task, ruleset_name=run_name)

                request = GenerationRequest(
                    question=build_question(task, subrun)
                )
                answer: GenerationResult = runtime.generate(request, policy)
                task_result = RulesetEvaluationResult(task.row, answer=answer)

                if task_result.ok:
                    correct += 1

                printer.task_end(
                    task,
                    ruleset_name=run_name,
                    task_result=task_result,
                )
                task_results.append(task_result)
                progress.accuracy(correct / len(task_results))
                save_result(subrun.name, variant, task_result)

        except BaseException:
            # Preserve whatever completed before the interruption, then fail normally.
            download_results(f"{i + 1}-{variant}-partial")
            raise

        results[(subrun.name, variant)] = tuple(task_results)
        download_results(f"{i + 1}-{variant}")

0. Evaluating trajectory-editing-orl-ssnhl-acute-system-only:


trajectory-editing-orl-ssnhl-acute-system-only:   0%|          | 0/1000 [00:00<?, ?case/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/lavrenko/group-beam-search:
- custom_generate/beam_search.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] A new version of the following files was downloaded from https://huggingface.co/lavrenko/group-beam-search:
- custom_generate/generate.py
- beam_search.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

1. Evaluating trajectory-editing-orl-conductive-acute-duplicated-after-query:


trajectory-editing-orl-conductive-acute-duplicated-after-query:   0%|          | 0/1000 [00:00<?, ?case/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Accuracy and runtime behavior by instruction-placement condition

For each clinical subrun, the same selected cases are evaluated under two
otherwise identical conditions:

- **system-only** — the Answer Engineering instruction appears only in the
  system message;
- **duplicated-after-query** — the same instruction is also appended after the
  clinical question.

The report includes exact-match accuracy, mean runtime per case, rule-trigger
and intervention rates, avoid-rule activity, and alternative-trajectory search
effort. “Alternatives tried” counts generated alternatives actually evaluated,
averaged over generated avoid-probe sets. The 80% resolution budget and
exhaustion rate follow the runtime telemetry definitions used in the paper.

In [7]:
from ae_paper_reproduction.telemetry.ssnhl_experiment import summarize_telemetry


for subrun in subruns:
    print(f"\n{subrun.name}")

    for variant in request_variants:
        rs = results[(subrun.name, variant)]
        t = summarize_telemetry([
            r.ae_telemetry
            for r in rs
            if r.ae_telemetry is not None
        ])

        alternatives_tried = (
            t.consumed_generated_probes_total
            / t.probe_sets_generated_total
            if t.probe_sets_generated_total else 0
        )

        correct = sum(r.ok for r in rs)

        print(
            f"  {variant}\n"
            f"    accuracy: {correct}/{len(rs)} = {correct / len(rs):.1%}\n"
            f"    runtime: {t.runtime_seconds_per_case:.2f} sec/case\n"
            f"    triggers: {t.avg_triggers_per_case:.2f}/case\n"
            f"    interventions: {t.avg_interventions_per_case:.2f}/case\n"
            f"    avoid interventions: {t.avoid_interventions_per_case:.2f}/case\n"
            f"    avoid probe episodes: {t.avoid_probe_episodes_per_case:.2f}/case\n"
            f"    alternatives tried: {alternatives_tried:.2f}/probe set\n"
            f"    alternatives for 50% resolution: "
            f"{t.probe_budget_for_50_coverage}\n"
            f"    alternatives for 80% resolution: "
            f"{t.probe_budget_for_80_coverage}\n"
            f"    ran out of alternatives: {t.not_enough_probes_share:.1%}"
        )


trajectory-editing-orl-ssnhl-acute
  system-only
    accuracy: 842/1000 = 84.2%
    runtime: 6.94 sec/case
    triggers: 24.57/case
    interventions: 17.49/case
    avoid interventions: 15.72/case
    avoid probe episodes: 15.38/case
    alternatives tried: 3.70/probe set
    alternatives for 50% resolution: 2
    alternatives for 80% resolution: 6
    ran out of alternatives: 8.2%
  duplicated-after-query
    accuracy: 971/1000 = 97.1%
    runtime: 5.65 sec/case
    triggers: 8.30/case
    interventions: 7.54/case
    avoid interventions: 5.62/case
    avoid probe episodes: 5.61/case
    alternatives tried: 2.56/probe set
    alternatives for 50% resolution: 2
    alternatives for 80% resolution: 3
    ran out of alternatives: 0.4%

trajectory-editing-orl-conductive-acute
  system-only
    accuracy: 786/1000 = 78.6%
    runtime: 3.74 sec/case
    triggers: 1.62/case
    interventions: 1.61/case
    avoid interventions: 1.40/case
    avoid probe episodes: 1.40/case
    alternatives t

## Paired case-level changes

Because both conditions evaluate the same cases in the same order, we can also
compare their outcomes case by case.

- **improved** counts cases that were incorrect with the system-only prompt but
  correct when the instruction was repeated after the query;
- **degraded** counts cases that were correct with the system-only prompt but
  incorrect after duplication;
- **net** is `improved - degraded`.

This paired comparison shows whether the aggregate accuracy difference reflects
a consistent directional change rather than unrelated successes and failures.

In [8]:
for subrun in subruns:
    baseline = results[(subrun.name, "system-only")]
    duplicated = results[(subrun.name, "duplicated-after-query")]

    improved = sum(
        not before.ok and after.ok
        for before, after in zip(baseline, duplicated, strict=True)
    )
    degraded = sum(
        before.ok and not after.ok
        for before, after in zip(baseline, duplicated, strict=True)
    )

    print(
        f"{subrun.name}: "
        f"improved={improved}, degraded={degraded}, "
        f"net={improved - degraded:+d}"
    )

download_results("final")

trajectory-editing-orl-ssnhl-acute: improved=154, degraded=25, net=+129
trajectory-editing-orl-conductive-acute: improved=153, degraded=201, net=-48


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Answer Engineering Rules

## Run: trajectory-editing

## Mode: trajectory

## Paper Role: primary

## Variant: trajectory

- orl-ssnhl-acute
- orl-conductive-acute

## System Prompt

You are an experienced physician. Interpret the key findings before drawing conclusions. Synthesize them into a clinical assessment, then state the most appropriate management decision. Always give a concrete management action that can be started now when the case allows. Do not make referral the primary recommendation. Prioritize time-sensitive conditions when supported by the findings. Use only what is explicitly stated and keep reasoning concise.

## Replace: sensorineural hearing loss

With:

- sudden sensorineural hearing loss
- SSNHL

Prefix:

- sudden
- abrupt
- acute
- rapid onset
- within 1-72 hours
- noticed 1-72 hours

## After: SSNHL

Add:

- This condition requires urgent treatment.
- Prompt treatment is indicated.
- Treatment should be initiated without delay.

## Avoid (prefix clause): rationalization of a diagnosis with tests

Scope: all

Prefix (any):

- conductive
- sensorineural
- stroke
- otitis
- allergic reaction
- autoimmune
- otolaryngologist
- ENT

Postfix:

- test
- testing

Fallback: The test results shall be analyzed carefully.

## Avoid (last clause): contralateral conductive inference Weber

Scope: all

Prefix:

- Weber | forehead
- left || right

Postfix:

- right || left
- conductive

Fallback:

- The Weber finding should be interpreted in relation to the affected ear.
- The Weber result should be interpreted with respect to both ears.
- The Weber lateralization should be interpreted relative to the side of symptoms.

## Avoid (last clause): Rinne positive then conductive

Scope: all

Prefix:

- Rinne
- positive

Postfix: conductive

Fallback: , which shall be analyzed carefully together with other tests.

## Avoid (last clause): explicit Rinne positive then conductive

Scope: all

Prefix: air conduction is greater than bone conduction

Postfix: conductive

Fallback: , which should be interpreted together with other tests.

## Avoid (last sentence): incomplete laterality then diagnosis

Scope: all

Prompt (all):

- left
- right

Prefix (incomplete):

- left
- right

Postfix (any):

- conductive
- sensorineural
- stroke
- otitis
- allergic reaction
- autoimmune
- otolaryngologist
- ENT

Fallback:

- Weber and Rinne findings in both left and right ears should be interpreted first.
- Tuning fork tests in both left and right ears must be evaluated before diagnosis.
- The Weber and Rinne results in both left and right ears should be analyzed first.

## Avoid (last sentence): no fork then diagnosis

Scope: all

Prefix (none):

- fork
- Weber
- Rinne

Postfix (any):

- conductive
- sensorineural
- stroke
- otitis
- allergic reaction
- autoimmune
- otolaryngologist
- ENT

Fallback:

- Weber and Rinne findings should be interpreted first.
- Tuning fork tests must be evaluated before diagnosis.
- The Weber and Rinne results should be analyzed first.

## Avoid (last clause): hearing loss and positive Rinne then normal

Scope: all

Prefix:

- hearing loss
- Rinne
- positive

Postfix: normal

Fallback: .

## Avoid (last clause): hearing loss and explicit positive Rinne then normal

Scope: all

Prefix:

- hearing loss
- Rinne
- air conduction is greater than bone conduction

Postfix: normal

Fallback: .

---